In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# ----------------------------------
# CLIP-Driven Dataset for KD
# ----------------------------------
class CIFAR10WithCLIPEmbeds(Dataset):
    def __init__(self, split="train"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").eval().to(self.device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

        self.resize = transforms.Resize((224, 224))
        self.tensorify = transforms.ToTensor()

        self.dataset = datasets.CIFAR10(root="./data", train=(split == "train"), download=True, transform=None)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        pil_img, _ = self.dataset[idx]
        resized_img = self.resize(pil_img)

        # Prepare input for CLIP
        inputs = self.processor(text=[""], images=resized_img, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            clip_output = self.clip_model(**inputs)
            clip_embed = clip_output.image_embeds.squeeze(0).cpu()  # [768]

        img_tensor = self.tensorify(resized_img)  # [3, 224, 224]
        return img_tensor, clip_embed


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.skip = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.skip(x)
        out = self.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.prep = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        self.layer1 = nn.Sequential(
            BasicBlock(64, 64, stride=1),
            BasicBlock(64, 64, stride=1)
        )
        self.layer2 = nn.Sequential(
            BasicBlock(64, 128, stride=2),
            BasicBlock(128, 128, stride=1)
        )
        self.layer3 = nn.Sequential(
            BasicBlock(128, 256, stride=2),
            BasicBlock(256, 256, stride=1)
        )
        self.layer4 = nn.Sequential(
            BasicBlock(256, 512, stride=2),
            BasicBlock(512, 512, stride=1)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return x

class ProjectionHead(nn.Module):
    def __init__(self, in_dim=512, out_dim=768):
        super().__init__()
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.proj(x)


In [ ]:
import os
import torch
from tqdm import tqdm

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = CIFAR10WithCLIPEmbeds(split="train")
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)

backbone = ResNet18Backbone().to(device)
projection = ProjectionHead(in_dim=768, out_dim=10).to(device)

params = list(backbone.parameters()) + list(projection.parameters())
optimizer = torch.optim.Adam(params, lr=1e-3)
loss_fn = nn.MSELoss()

# Directory to save checkpoint
save_dir = "./checkpoints"
os.makedirs(save_dir, exist_ok=True)
checkpoint_path = os.path.join(save_dir, "resnet_clip_kd_checkpoint.pth")
loss_log_path = os.path.join(save_dir, "per_batch_losses.pt")

# For logging
loss_log = []

# Train
start_epoch = 0
num_epochs = 4

for epoch in range(start_epoch, start_epoch + num_epochs):
    backbone.train()
    projection.train()

    running_loss = 0.0
    epoch_losses = []  # Store loss.item() for each batch

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for img_batch, clip_embed_batch in progress_bar:
        img_batch = img_batch.to(device)
        clip_embed_batch = clip_embed_batch.to(device)

        feats = backbone(img_batch)
        preds = projection(feats)
        loss = loss_fn(preds, clip_embed_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        running_loss += batch_loss
        epoch_losses.append(batch_loss)

        progress_bar.set_postfix(loss=batch_loss)

    avg_loss = running_loss / len(dataloader)
    loss_log.append(epoch_losses)  # Save this epoch's batch losses

    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint after each epoch
    torch.save({
        "epoch": epoch + 1,
        "backbone_state_dict": backbone.state_dict(),
        "projection_state_dict": projection.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": avg_loss,
    }, checkpoint_path)

    # Save per-batch loss log
    torch.save(loss_log, loss_log_path)

    print(f"Checkpoint and batch losses saved to {save_dir}")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

# -------------------------
# Configs & Setup
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "/kaggle/input/cifar10-embeddings/checkpoints/resnet_clip_kd_checkpoint.pth"  # replace with actual path
num_classes = 10
batch_size = 32
num_epochs = 5

# -------------------------
# DataLoader - CIFAR-10 Test Set (can be used for train too)
# -------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


class ClassificationHead(nn.Module):
    def __init__(self, input_dim=512, num_classes=10):
        super().__init__()
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.classifier(x)

backbone = ResNet18Backbone().to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
backbone.load_state_dict(checkpoint["backbone_state_dict"])
backbone.eval()

for param in backbone.parameters():
    param.requires_grad = False

# -------------------------
# Initialize Classification Head
# -------------------------
classification_head = ClassificationHead(input_dim=512, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classification_head.parameters(), lr=1e-4)

# -------------------------
# Training Loop
# -------------------------
for epoch in range(num_epochs):
    classification_head.train()
    total_loss, correct, total = 0.0, 0, 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    for imgs, labels in progress_bar:
        imgs, labels = imgs.to(device), labels.to(device)

        with torch.no_grad():
            features = backbone(imgs)

        logits = classification_head(features)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=loss.item(), acc=100 * correct / total)

    print(f"[Epoch {epoch+1}] Train Loss: {total_loss:.4f} | Train Acc: {100 * correct / total:.2f}%")

# -------------------------
# Evaluation on Test Set
# -------------------------
classification_head.eval()
correct, total = 0, 0

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Evaluating"):
        imgs, labels = imgs.to(device), labels.to(device)
        feats = backbone(imgs)
        logits = classification_head(feats)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")
